# Sensitivity analysis in ModernDiD and R

This notebook is generated from the [Sensitivity analysis in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_honest.html) page of the ModernDiD documentation.

This comparison checks whether [`honest_did`](https://moderndid.readthedocs.io/en/stable/api/generated/honestdid/moderndid.honest_did.html) and R's
`HonestDiD` package return the same robust confidence intervals for the
Medicaid expansion event study in the
[sensitivity analysis example](https://moderndid.readthedocs.io/en/stable/user_guide/example_honest_did.html#example-honest-did). `HonestDiD` is the
authors' own implementation of
[Rambachan and Roth (2023)](https://doi.org/10.1093/restud/rdad018), the paper
that ModernDiD's sensitivity analysis follows. Both packages receive the same
event-study coefficients and the same covariance matrix. From those inputs the
page compares the original interval under exact parallel trends and the robust
intervals under smoothness and relative magnitudes restrictions, for every
method the two packages share.

The original interval, the conditional test, and the conditional FLCI hybrid
agree to rounding error. The page traces each of the two remaining gaps to its
source. R's fixed-length intervals sit up to 7.5e-5 from ModernDiD's because R
estimates a quantile of the folded normal distribution by simulation. Once R
uses the exact quantile instead, the two packages agree to within 5.4e-7.
Because each package simulates the critical value of the conditional least
favorable hybrid with its own random draws, that method's intervals can differ
by one step of the grid of candidate effects. With only two placebo estimates,
R's lower bound on $M$ also stops with an indexing error where ModernDiD returns
a value.

Since the R functions run inside this Python session through `rpy2`, they read
the exact arrays that ModernDiD builds. The section on
[running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running) explains how to
install the R environment and rerun these cells.

> **What the comparison holds fixed**
>
> Every gap on this page comes from the sensitivity analysis rather than from
> the event study. The [staggered comparison](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_staggered.html#r-comparisons-staggered)
> checks the estimator behind the event study against R's `did` package on the
> minimum wage data.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.4.0", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-honestdid"], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

In [ ]:
import polars as pl
import rpy2.robjects as ro

import moderndid as did

In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import numpy2ri
from rpy2.robjects.packages import importr

with ro.default_converter.context():
    importr("HonestDiD")

## The event study both packages read

[`load_ehec`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_ehec.html) loads the panel of the sensitivity analysis
example, 46 states observed in every year from 2008 through 2019. Of those
states, 30 expanded Medicaid under the Affordable Care Act between 2014 and
2019. The outcome `dins` is the share of low-income adults without children who
had health insurance. The example's event study compares each cohort with the
states that haven't expanded yet and measures every estimate from event
time −1, the base year under `base_period="universal"`. Its window keeps the
five years on either side of expansion.

In [ ]:
# States that hadn't expanded by 2019 get a 0, the value att_gt reads as never treated.
data = did.load_ehec().with_columns(expansion_year=pl.col("yexp2").fill_null(0).cast(pl.Int64))

# The att_gt and aggte calls from the sensitivity analysis example.
result = did.att_gt(
    data,
    yname="dins",
    tname="year",
    idname="stfips",
    gname="expansion_year",
    control_group="notyettreated",
    base_period="universal",
    cband=False,
)
event_study = did.aggte(result, type="dynamic", min_e=-5, max_e=5)
print(event_study)

Besides the base year at event time −1, the event study holds four placebo
estimates at event times −5 to −2 and six effects at 0 to 5. Since every
estimate is measured from the base year, [`honest_did`](https://moderndid.readthedocs.io/en/stable/api/generated/honestdid/moderndid.honest_did.html) drops
that row and passes the ten remaining estimates to the sensitivity analysis.
For an event study estimated without `clustervars`, as this one is, their
covariance is the outer product of their influence functions over the 46
states divided by 46 squared. We build both inputs the same way and hand them
to R.

> **HonestDiD's wrapper ignores clusters**
>
> For an event study estimated with `clustervars`, [`honest_did`](https://moderndid.readthedocs.io/en/stable/api/generated/honestdid/moderndid.honest_did.html)
> sums the influence functions within each cluster before it forms the
> covariance. The `honest_did` method that `HonestDiD` provides for event
> studies from R's `did` package never sums within clusters ([line 52 of
> `R/honest_did.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/honest_did.R#L52)).
> For a clustered event study, report ModernDiD's intervals, since R's wrapper
> starts from a covariance that treats units in one cluster as independent.

In [ ]:
def honest_inputs(study):
    """Return the inputs that honest_did builds from an event study without clusters."""
    # honest_did leaves out the base year at event time -1, the reference for every estimate.
    keep = study.event_times != -1
    influence = study.influence_func[:, keep]
    return study.att_by_event[keep], influence.T @ influence / influence.shape[0] ** 2


betahat, sigma = honest_inputs(event_study)
with (ro.default_converter + numpy2ri.converter).context():
    ro.globalenv["betahat"] = betahat
    ro.globalenv["sigma"] = sigma

## The original interval and fixed-length intervals

We start with the example's own analysis of the effect in the year of
expansion, smoothness bounds $M$ from 0 to 0.03 with fixed-length confidence
intervals (FLCI). Besides one robust interval for each $M$,
[`honest_did`](https://moderndid.readthedocs.io/en/stable/api/generated/honestdid/moderndid.honest_did.html) returns the original interval, the estimate's 95
percent interval under exact parallel trends. In R, `constructOriginalCS`
computes the original interval and `createSensitivityResults` the robust ones.

In [ ]:
# The example's smoothness analysis of the effect in the year of expansion.
m_vec = [0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
smoothness = did.honest_did(
    event_study, event_time=0, sensitivity_type="smoothness", m_vec=m_vec, method="FLCI"
)
print(smoothness.robust_ci)

In [ ]:
# The first four coefficients are placebo estimates and the last six are effects.
with ro.default_converter.context():
    ro.globalenv["m_vec"] = ro.FloatVector(m_vec)
    ro.r("""
        r_original <- HonestDiD::constructOriginalCS(
            betahat, sigma, numPrePeriods = 4, numPostPeriods = 6
        )
        r_flci <- HonestDiD::createSensitivityResults(
            betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, method = "FLCI", Mvec = m_vec
        )
    """)

In [ ]:
def side_by_side(labels, python_bounds, r_bounds):
    """Set each interval from ModernDiD beside the one from R and subtract R's ends."""
    table = labels.copy()
    table["ModernDiD lb"] = python_bounds[:, 0]
    table["R lb"] = r_bounds[:, 0]
    table["lb difference"] = table["ModernDiD lb"] - table["R lb"]
    table["ModernDiD ub"] = python_bounds[:, 1]
    table["R ub"] = r_bounds[:, 1]
    table["ub difference"] = table["ModernDiD ub"] - table["R ub"]
    return table


with ro.default_converter.context():
    r_original = np.asarray(ro.r("c(r_original$lb, r_original$ub)"))
    r_flci = np.asarray(ro.r("cbind(r_flci$lb, r_flci$ub)"))

# The original interval assumes exact parallel trends and has no smoothness bound.
python_flci = np.vstack(
    [
        [smoothness.original_ci.lb, smoothness.original_ci.ub],
        smoothness.robust_ci.select("lb", "ub").to_numpy(),
    ]
)
flci_comparison = side_by_side(
    pd.DataFrame({"method": ["Original", *smoothness.robust_ci["method"]], "m": [np.nan, *m_vec]}),
    python_flci,
    np.vstack([r_original, r_flci]),
)
flci_comparison

The original interval matches to rounding error, since both packages form it
as the same estimate plus or minus the same normal quantile times the same
standard error. The fixed-length intervals differ by 1.2e-5 to 1.5e-5 at every
$M$ except 0.005. At that bound the gap grows to 7.5e-5 at the upper end.


### A simulated quantile in R

Each fixed-length interval is a weighted sum of the ten estimates plus or minus
a half-length. That half-length is the weighted sum's standard deviation times
a quantile of the folded normal distribution, the distribution of $|Z + t|$ for
a standard normal $Z$ and the ratio $t$ of the worst-case bias to that standard
deviation. Where [`folded_normal_quantile`](https://moderndid.readthedocs.io/en/stable/api/generated/honestdid/moderndid.folded_normal_quantile.html) solves for this
quantile exactly, `HonestDiD` reads it off 1,000,000 simulated draws with seed 0
([lines 97 to 104 of `R/flci.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/flci.R#L97-L104)).
Below are the two quantiles at zero bias, their ratio, and the ratio of the
interval lengths at $M = 0$.

In [ ]:
# R simulates the quantile at zero bias and ModernDiD solves for it exactly.
with ro.default_converter.context():
    simulated = ro.r("HonestDiD:::.qfoldednormal(p = 0.95, mu = 0)")[0]
exact = did.folded_normal_quantile(0.95, mu=0.0)

In [ ]:
# At M = 0 the two packages choose estimators with the same standard deviation.
at_zero = flci_comparison[flci_comparison["m"] == 0].iloc[0]
r_length = at_zero["R ub"] - at_zero["R lb"]
python_length = at_zero["ModernDiD ub"] - at_zero["ModernDiD lb"]
print(f"quantile at zero bias: {simulated:.5f} in R, {exact:.5f} in ModernDiD")
print(f"ratio of the quantiles: {simulated / exact:.5f}")
print(f"ratio of the interval lengths at M = 0: {r_length / python_length:.5f}")

At zero bias the exact quantile is 1.95996, the 97.5th percentile of the
standard normal distribution. R's simulation gives 1.96224 instead, about 0.12
percent more than the exact value. Because both packages choose estimators
with the same standard deviation at $M = 0$, R's interval there is longer by
that same ratio. Below, `createSensitivityResults` runs again with the exact
quantile in place of R's simulated one. The rest of the page goes back to R's
simulated quantile.

In [ ]:
# Replace R's simulated quantile with the exact one, rerun the intervals, and restore R's function.
with ro.default_converter.context():
    ro.r("""
        simulated_quantile <- HonestDiD:::.qfoldednormal
        exact_quantile <- function(p, mu = 0, sd = 1, numSims = 10^6, seed = 0) {
            vapply(abs(mu) / sd, function(shift) {
                gap <- function(x) pnorm(x - shift) - pnorm(-x - shift) - p
                uniroot(gap, c(0, shift + 10), tol = 1e-14)$root
            }, numeric(1)) * sd
        }
        assignInNamespace(".qfoldednormal", exact_quantile, ns = "HonestDiD")
        r_flci_exact <- HonestDiD::createSensitivityResults(
            betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, method = "FLCI", Mvec = m_vec
        )
        assignInNamespace(".qfoldednormal", simulated_quantile, ns = "HonestDiD")
    """)
    r_flci_exact = np.asarray(ro.r("cbind(r_flci_exact$lb, r_flci_exact$ub)"))

flci_exact_comparison = side_by_side(
    smoothness.robust_ci.select("method", "m").to_pandas(),
    smoothness.robust_ci.select("lb", "ub").to_numpy(),
    r_flci_exact,
)
exact_gap = flci_exact_comparison[["lb difference", "ub difference"]].abs().to_numpy().max()
print(f"largest gap with the exact quantile in R: {exact_gap:.1e}")

With the exact quantile, R's fixed-length intervals match ModernDiD's to within
5.4e-7 at every $M$. The swap also closes the larger gap that the first table
shows at $M = 0.005$. With its simulated quantile, R's search for the shortest
interval at that bound stops at the least biased estimator. In exact terms that
estimator's interval is 1.2e-5 longer than the one ModernDiD finds. The largest
remaining gap sits at $M = 0$ and comes from R's solvers rather than the
quantile.

> **Solver tolerance in R**
>
> At $M = 0$ the shortest interval comes from the estimator with the smallest
> variance, since every estimator either package considers is unbiased there.
> ModernDiD's center matches the exact center of that estimator to within 2e-11.
> R's solvers stop instead at nearby weights whose standard deviation matches
> that estimator's to within 1e-10 and whose center sits 5.4e-7 away.

## Test inversions on a shared grid

The conditional test (`"Conditional"`), the conditional FLCI hybrid (`"C-F"`),
and the conditional least favorable hybrid (`"C-LF"`) each check every
candidate effect on a grid. The interval a method reports runs from the
smallest candidate it accepts to the largest. To compare the tests rather than
the grids, we give both packages the same 301 candidates from −0.03 to 0.12,
one every 0.0005. Since `createSensitivityResults` takes no grid, the R side
calls `computeConditionalCS_DeltaSD` for each value of $M$ and
`computeConditionalCS_DeltaSDB` for the example's check with a positive bias.

> **Pass the same grid to both packages**
>
> For the conditional test and the least favorable hybrid, ModernDiD centers its
> default grid on the identified set that the estimates imply. `HonestDiD`
> centers both grids on zero however far the estimates sit from it. Runs that
> keep each package's default grid therefore report ends taken from different
> candidates.

In [ ]:
# Each R helper returns the smallest and largest candidate that its test accepts at one value of M.
with ro.default_converter.context():
    ro.r("""
        smoothness_cs <- function(m, flag, seed = 0) {
            cs <- HonestDiD::computeConditionalCS_DeltaSD(
                betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, M = m, hybrid_flag = flag,
                gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12, seed = seed
            )
            range(cs$grid[cs$accept == 1])
        }
        positive_bias_cs <- function(m) {
            cs <- HonestDiD::computeConditionalCS_DeltaSDB(
                betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, M = m, hybrid_flag = "FLCI",
                biasDirection = "positive", gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12
            )
            range(cs$grid[cs$accept == 1])
        }
    """)

# Each test's method and sign in honest_did, then the R call that runs the same test.
tests = [
    ("Conditional", None, 't(sapply(m_vec, smoothness_cs, flag = "ARP"))'),
    ("C-F", None, 't(sapply(m_vec, smoothness_cs, flag = "FLCI"))'),
    ("C-LF", None, 't(sapply(m_vec, smoothness_cs, flag = "LF"))'),
    ("C-F", "positive", "t(sapply(m_vec, positive_bias_cs))"),
]
tables = []
for method, bias, r_call in tests:
    python_ci = did.honest_did(
        event_study,
        event_time=0,
        sensitivity_type="smoothness",
        m_vec=m_vec,
        method=method,
        bias_direction=bias,
        grid_lb=-0.03,
        grid_ub=0.12,
        grid_points=301,
    ).robust_ci
    with ro.default_converter.context():
        r_bounds = np.asarray(ro.r(r_call))
    labels = python_ci.select("method", "delta", "m").to_pandas()
    tables.append(side_by_side(labels, python_ci.select("lb", "ub").to_numpy(), r_bounds))
test_comparison = pd.concat(tables, ignore_index=True)
test_comparison

The conditional test and the conditional FLCI hybrid accept the same candidates
in both packages at every $M$, in the DeltaSD rows without a sign and in the
DeltaSDPB rows with a positive bias. Although the hybrid's first stage uses
R's simulated quantile too, at the 99.5 percent level, the small shift moves no
candidate across the edge of a confidence set here. In the least favorable
hybrid, ModernDiD's interval is one step narrower at both ends up to
$M = 0.02$. At 0.025 and 0.03 only its upper end sits one step lower.


### Seeds and the least favorable hybrid

Before it runs the conditional test, the least favorable hybrid rejects any
candidate whose test statistic exceeds a first-stage critical value. Each
package simulates that critical value as the 99.5th percentile of the statistic
over 1,000 simulated normal draws. Since ModernDiD draws with NumPy's generator
and `HonestDiD` with R's, the two critical values differ by simulation noise.
The table below lists the range of each end when each package repeats this
hybrid under seeds 0 to 9.

In [ ]:
# The least favorable hybrid in each package under seeds 0 to 9.
runs = []
for seed in range(10):
    python_ci = did.honest_did(
        event_study,
        event_time=0,
        sensitivity_type="smoothness",
        m_vec=m_vec,
        method="C-LF",
        seed=seed,
        grid_lb=-0.03,
        grid_ub=0.12,
        grid_points=301,
    ).robust_ci.to_pandas()
    with ro.default_converter.context():
        ro.globalenv["seed"] = seed
        r_bounds = np.asarray(ro.r('t(sapply(m_vec, smoothness_cs, flag = "LF", seed = seed))'))
    runs.append(python_ci[["m", "lb", "ub"]].assign(package="ModernDiD"))
    runs.append(pd.DataFrame({"m": m_vec, "lb": r_bounds[:, 0], "ub": r_bounds[:, 1], "package": "R"}))
pd.concat(runs).groupby(["m", "package"]).agg(["min", "max"])

Across the ten seeds, ModernDiD's upper end at $M = 0$ moves by two steps of
the grid and every other end by one. R's ends lie within ModernDiD's range at
every value of $M$. The one-step gaps in the least favorable hybrid are
therefore the noise of a simulation from 1,000 draws rather than a difference
between the tests.

> **The seed argument in HonestDiD**
>
> In the R rows, every seed gives the same two ends of each interval. With more
> than one effect after treatment, `HonestDiD` 0.2.6 simulates the least
> favorable critical value in `.ARP_computeCI`. Since that function passes no
> seed to the simulation
> ([lines 737 to 740 of `R/arp-nuisance.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/arp-nuisance.R#L737-L740)),
> the draws always use the default seed of 0 whatever `seed` you pass.

## Relative magnitudes

Under relative magnitudes, both packages offer two methods, the conditional
test and the default least favorable hybrid. Neither offers fixed-length
intervals for this restriction, because such an interval would have to be
infinitely long. Since `createSensitivityResults_relativeMagnitudes` accepts a
grid, both packages test the same 301 candidates for each $\bar{M}$ from 0.5 to
3, the example's grid.

In [ ]:
# Both packages test the example's relative magnitudes bounds on the shared candidates.
m_bar_vec = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
with ro.default_converter.context():
    ro.globalenv["m_bar_vec"] = ro.FloatVector(m_bar_vec)

tables = []
for method in ["C-LF", "Conditional"]:
    python_ci = did.honest_did(
        event_study,
        event_time=0,
        sensitivity_type="relative_magnitude",
        m_bar_vec=m_bar_vec,
        method=method,
        grid_lb=-0.03,
        grid_ub=0.12,
        grid_points=301,
    ).robust_ci
    with ro.default_converter.context():
        ro.globalenv["method"] = method
        r_bounds = np.asarray(
            ro.r("""
                r_rm <- HonestDiD::createSensitivityResults_relativeMagnitudes(
                    betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, method = method,
                    Mbarvec = m_bar_vec, gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12
                )
                cbind(r_rm$lb, r_rm$ub)
            """)
        )
    labels = python_ci.select("method", "delta", "Mbar").to_pandas()
    tables.append(side_by_side(labels, python_ci.select("lb", "ub").to_numpy(), r_bounds))
rm_comparison = pd.concat(tables, ignore_index=True)
rm_comparison

The conditional test accepts the same candidates in both packages at every
$\bar{M}$. In the least favorable hybrid, ModernDiD's interval reaches one step
further at two ends. Its upper end sits one step higher at $\bar{M} = 2.5$ and
its lower end one step lower at $\bar{M} = 3$. Because R ignores its seed here,
only ModernDiD runs again at those two bounds over the same ten seeds.

In [ ]:
# ModernDiD's least favorable hybrid under seeds 0 to 9 at the two bounds where the packages differ.
runs = [
    did.honest_did(
        event_study,
        event_time=0,
        sensitivity_type="relative_magnitude",
        m_bar_vec=[2.5, 3.0],
        method="C-LF",
        seed=seed,
        grid_lb=-0.03,
        grid_ub=0.12,
        grid_points=301,
    ).robust_ci.to_pandas()
    for seed in range(10)
]
pd.concat(runs).groupby("Mbar")[["lb", "ub"]].agg(["min", "max"])

Over the ten seeds, ModernDiD's upper end at $\bar{M} = 2.5$ runs from 0.0850
to 0.0860 and its lower end at $\bar{M} = 3$ from −0.0075 to −0.0065. R's ends
of 0.0855 and −0.0070 lie inside both ranges, as they did under smoothness.

## Shorter pre-treatment windows

The definitions of the restrictions fix how many placebo estimates each one
needs. The relative magnitudes restriction compares every yearly change from
the base year on with the largest yearly change before treatment. With a single
placebo estimate, that largest change is the step from event time −2 to the
base year. The linear-trend bound that `bound="deviation from linear trend"`
selects compares changes in slope instead. It needs two placebo estimates to
form one change in slope before treatment. Both packages therefore run relative
magnitudes with one placebo estimate and the linear-trend bound with two. We
trim the event study to those minimums with `min_e` and run the conditional
test under each restriction.

In [ ]:
# Event studies with one and with two placebo estimates before the base year.
one_placebo = did.aggte(result, type="dynamic", min_e=-2, max_e=5)
two_placebos = did.aggte(result, type="dynamic", min_e=-3, max_e=5)

tables = []
for study, bound, num_pre in [
    (one_placebo, "deviation from parallel trends", 1),
    (two_placebos, "deviation from linear trend", 2),
]:
    python_ci = did.honest_did(
        study,
        event_time=0,
        sensitivity_type="relative_magnitude",
        m_bar_vec=m_bar_vec,
        method="Conditional",
        bound=bound,
        grid_lb=-0.03,
        grid_ub=0.12,
        grid_points=301,
    ).robust_ci
    short_betahat, short_sigma = honest_inputs(study)
    with (ro.default_converter + numpy2ri.converter).context():
        ro.globalenv["short_betahat"] = short_betahat
        ro.globalenv["short_sigma"] = short_sigma
        ro.globalenv["bound"] = bound
        ro.globalenv["num_pre"] = num_pre
        r_bounds = ro.r("""
            r_short <- HonestDiD::createSensitivityResults_relativeMagnitudes(
                short_betahat, short_sigma, numPrePeriods = num_pre, numPostPeriods = 6,
                bound = bound, method = "Conditional", Mbarvec = m_bar_vec,
                gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12
            )
            cbind(r_short$lb, r_short$ub)
        """)
    labels = python_ci.select("delta", "Mbar").to_pandas()
    tables.append(side_by_side(labels, python_ci.select("lb", "ub").to_numpy(), r_bounds))
short_comparison = pd.concat(tables, ignore_index=True)
short_comparison

Both restrictions give the same intervals in the two packages at every
$\bar{M}$.


### The lower bound on M

The smoothness analysis has a diagnostic that needs two placebo estimates as
well, enough for one change in slope. Its lower bound on $M$ is the smallest $M$
that the placebo estimates don't reject. Both ModernDiD's
`compute_delta_sd_lowerbound_m` and R's `DeltaSD_lowerBound_Mpre` check for that
minimum before they compute anything. The table below gives the bound in each
package with two placebo estimates and with three.

In [ ]:
# The lower bound on M from two and from three placebo estimates.
three_placebos = did.aggte(result, type="dynamic", min_e=-4, max_e=5)
rows = []
for study, num_pre in [(two_placebos, 2), (three_placebos, 3)]:
    lower_betahat, lower_sigma = honest_inputs(study)
    with (ro.default_converter + numpy2ri.converter).context():
        ro.globalenv["lower_betahat"] = lower_betahat
        ro.globalenv["lower_sigma"] = lower_sigma
        ro.globalenv["num_pre"] = num_pre
        r_lower = ro.r("""
            tryCatch(
                format(HonestDiD::DeltaSD_lowerBound_Mpre(
                    lower_betahat, lower_sigma, numPrePeriods = num_pre
                )),
                error = function(e) paste("error:", conditionMessage(e))
            )
        """)[0]
    python_lower = did.compute_delta_sd_lowerbound_m(lower_betahat, lower_sigma, num_pre_periods=num_pre)
    rows.append({"placebo estimates": num_pre, "ModernDiD": python_lower, "R": r_lower})
pd.DataFrame(rows)

With two placebo estimates, ModernDiD returns a lower bound of 0, since the
three points from event time −3 to the base year are consistent with a straight
line. R stops with an error instead, although its own check allows two. In
`HonestDiD` 0.2.6,
[line 48 of `R/ublbM_functions.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/ublbM_functions.R#L48)
loops over `1:(numPrePeriods-2)` to fill in the remaining changes in slope.
When `numPrePeriods` is 2, that sequence counts down from 1 to 0 rather than
coming out empty. Its first pass then writes three coefficients into a row of a
matrix that has only two columns. Both packages return the same lower bound of
0 from three placebo estimates.

## Tolerances and assertions

Each comparison above becomes an assertion that a later rerun must pass. The
original interval and every interval from the conditional test or the
conditional FLCI hybrid must match to 1e-12. The fixed-length intervals may
differ by 1e-4 with R's simulated quantile and by 1e-6 with the exact one. That
second tolerance leaves room above the 5.4e-7 shown earlier because R's solvers
can stop at other weights when you rerun the page. The least favorable hybrid
may differ by one step of the shared grid. The table lists the largest gap in
each comparison next to its tolerance.

In [ ]:
# The least favorable hybrid may differ by one step of the shared grid, 0.0005.
smoothness_lf = test_comparison["method"] == "C-LF"
rm_lf = rm_comparison["method"] == "C-LF"
checks = [
    ("original interval", flci_comparison.iloc[:1], 1e-12),
    ("FLCI, simulated quantile in R", flci_comparison.iloc[1:], 1e-4),
    ("FLCI, exact quantile in R", flci_exact_comparison, 1e-6),
    ("Conditional and C-F, smoothness", test_comparison[~smoothness_lf], 1e-12),
    ("C-LF, smoothness", test_comparison[smoothness_lf], 0.0005 + 1e-12),
    ("Conditional, relative magnitudes", rm_comparison[~rm_lf], 1e-12),
    ("C-LF, relative magnitudes", rm_comparison[rm_lf], 0.0005 + 1e-12),
    ("Conditional, shorter windows", short_comparison, 1e-12),
]

rows = []
for label, table, tolerance in checks:
    np.testing.assert_allclose(table["ModernDiD lb"], table["R lb"], rtol=0, atol=tolerance)
    np.testing.assert_allclose(table["ModernDiD ub"], table["R ub"], rtol=0, atol=tolerance)
    rows.append(
        {
            "comparison": label,
            "largest lb difference": table["lb difference"].abs().max(),
            "largest ub difference": table["ub difference"].abs().max(),
            "tolerance": tolerance,
        }
    )
pd.DataFrame(rows)

## Versions used

The output above came from the versions of Python, R, `rpy2`, and the two
packages listed below.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    versions = {
        "Python": python_version(),
        "ModernDiD": version("moderndid"),
        "rpy2": version("rpy2"),
        "R": ro.r("R.version.string")[0],
        "HonestDiD": ro.r('as.character(packageVersion("HonestDiD"))')[0],
    }
pd.DataFrame({"component": list(versions), "version": list(versions.values())})